# NREL PySAM (System Advisor Model) Feature Showcase

This notebook demonstrates the key features and capabilities of NREL's PySAM (Python System Advisor Model), which provides a Python interface to SAM's simulation capabilities for renewable energy systems.

## Table of Contents
1. [Introduction to PySAM](#introduction)
2. [Installation and Setup](#installation)
3. [Model Initialization Methods](#initialization)
4. [Solar PV System Modeling](#pv-modeling)
5. [Battery Storage Integration](#battery-storage)
6. [Financial Analysis](#financial)
7. [Wind Power Systems](#wind)
8. [Weather Data Integration](#weather)
9. [Sensitivity Analysis](#sensitivity)
10. [Advanced Features](#advanced)

---

## 1. Introduction to PySAM {#introduction}

PySAM is NREL's Python wrapper for the System Advisor Model (SAM) Simulation Core. It enables:

- **Renewable Energy Modeling**: Solar PV, wind, geothermal, biomass, and more
- **Energy Storage**: Battery systems and hybrid configurations
- **Financial Analysis**: LCOE, NPV, payback periods, and various ownership models
- **Weather Integration**: TMY data and resource assessment
- **Parametric Studies**: Sensitivity analysis and optimization

### Key Advantages
- Validated physics models from NREL
- Comprehensive financial modeling
- Python integration for automation and analysis
- Extensive default databases
- Open source and free to use

## 2. Installation and Setup {#installation}

### Installation
```bash
pip install NREL-PySAM
pip install NREL-PySAM-stubs  # For IDE autocompletion
```

### System Requirements
- Python 3.9-3.13
- 64-bit systems (Windows, macOS, Linux)
- ~500 MB disk space for full installation

In [ ]:
# Import core PySAM modules
import PySAM
import PySAM.Pvsamv1 as PV
import PySAM.Grid as Grid
import PySAM.Utilityrate5 as UtilityRate
import PySAM.Cashloan as CashLoan
import PySAM.Battery as Battery
import PySAM.Windpower as Wind

# Supporting libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

print(f"PySAM Version: {PySAM.__version__}")
print("Available modules:")
for module in sorted(dir(PySAM)):
    if not module.startswith('_') and hasattr(getattr(PySAM, module), 'default'):
        print(f"  - {module}")

## 3. Model Initialization Methods {#initialization}

PySAM offers four ways to initialize models, each with different use cases:

In [ ]:
# Method 1: "new" - Creates empty model instance
pv_new = PV.new()
print("Method 1 - new(): Creates empty model")
print(f"Number of assigned variables: {len([k for k in pv_new.export() if pv_new.export()[k] is not None])}")

# Method 2: "default" - Loads SAM default configuration
pv_default = PV.default("FlatPlatePVSingleOwner")
print("\nMethod 2 - default(): Loads SAM defaults")
print(f"Number of assigned variables: {len([k for k in pv_default.export() if pv_default.export()[k] is not None])}")
print(f"Default system capacity: {pv_default.SystemDesign.system_capacity} kW")

# Method 3: "from_existing" - Shares data between models
pv_shared = PV.from_existing(pv_default)
print("\nMethod 3 - from_existing(): Shares data with another model")
print(f"Shared system capacity: {pv_shared.SystemDesign.system_capacity} kW")

# Method 4: "wrap" - Creates from PySSC data
# This method is used when integrating with PySSC directly
print("\nMethod 4 - wrap(): Creates model from PySSC data structure (advanced use)")

### Exploring Model Structure

PySAM models are organized into logical groups of variables:

In [ ]:
# Explore the structure of a PV model
pv_model = PV.default("FlatPlatePVSingleOwner")

print("PV Model Variable Groups:")
for group_name in dir(pv_model):
    if not group_name.startswith('_') and hasattr(getattr(pv_model, group_name), '__dict__'):
        group = getattr(pv_model, group_name)
        if hasattr(group, '__dict__'):
            var_count = len([v for v in dir(group) if not v.startswith('_')])
            print(f"  {group_name}: {var_count} variables")

# Example: SystemDesign group variables
print("\nSystemDesign Group Variables (first 10):")
design_vars = [v for v in dir(pv_model.SystemDesign) if not v.startswith('_')]
for var in design_vars[:10]:
    try:
        value = getattr(pv_model.SystemDesign, var)
        print(f"  {var}: {value}")
    except:
        print(f"  {var}: [complex value]")

## 4. Solar PV System Modeling {#pv-modeling}

Let's create a comprehensive solar PV system model:

In [ ]:
# Create a detailed PV system model
pv = PV.default("FlatPlatePVResidential")

# System Design Parameters
pv.SystemDesign.system_capacity = 10.0  # 10 kW system
pv.SystemDesign.module_type = 0  # Standard silicon
pv.SystemDesign.array_type = 0   # Fixed - Open Rack
pv.SystemDesign.tilt = 30.0      # Tilt angle (degrees)
pv.SystemDesign.azimuth = 180.0  # South-facing
pv.SystemDesign.gcr = 0.4        # Ground coverage ratio

# Module specifications
pv.CECPerformanceModelWithModuleDatabase.cec_module_length = 1.65  # meters
pv.CECPerformanceModelWithModuleDatabase.cec_module_width = 1.0    # meters
pv.CECPerformanceModelWithModuleDatabase.cec_v_mp_ref = 31.1       # Vmp at STC
pv.CECPerformanceModelWithModuleDatabase.cec_i_mp_ref = 8.9        # Imp at STC
pv.CECPerformanceModelWithModuleDatabase.cec_v_oc_ref = 38.5       # Voc at STC
pv.CECPerformanceModelWithModuleDatabase.cec_i_sc_ref = 9.45       # Isc at STC

# Inverter specifications
pv.CECInverter.inv_snl_paco = 10000  # AC power rating (W)
pv.CECInverter.inv_snl_pdco = 10500  # DC input power at rated AC output (W)
pv.CECInverter.inv_snl_vdco = 310    # DC voltage at rated AC output (V)
pv.CECInverter.inv_snl_pso = 15      # DC power required for inverter startup (W)

# System losses
pv.SystemDesign.losses = 14.08  # Total system losses (%)

print("Solar PV System Configuration:")
print(f"System Capacity: {pv.SystemDesign.system_capacity} kW")
print(f"Tilt Angle: {pv.SystemDesign.tilt}°")
print(f"Azimuth: {pv.SystemDesign.azimuth}° (South)")
print(f"System Losses: {pv.SystemDesign.losses}%")
print(f"Inverter AC Rating: {pv.CECInverter.inv_snl_paco/1000} kW")

### Weather Data Integration

PySAM can work with various weather data formats:

In [ ]:
# Use built-in weather data or load from file
# For this example, we'll use the default weather file

# Option 1: Use default weather data (Phoenix, AZ)
weather_file = pv.SolarResource.solar_resource_file
print(f"Using weather file: {weather_file}")

# Option 2: Load custom TMY data (example path)
# pv.SolarResource.solar_resource_file = "/path/to/custom_weather.csv"

# Option 3: Create synthetic weather data
def create_synthetic_weather_data():
    """Create simple synthetic weather data for demonstration"""
    hours = 8760
    
    # Create realistic daily and seasonal patterns
    time = np.arange(hours)
    day_of_year = (time // 24) % 365
    hour_of_day = time % 24
    
    # Solar irradiance with seasonal and daily patterns
    seasonal_factor = 1 + 0.3 * np.cos(2 * np.pi * (day_of_year - 172) / 365)  # Peak in summer
    daily_pattern = np.maximum(0, np.cos(2 * np.pi * (hour_of_day - 12) / 24))
    ghi = seasonal_factor * daily_pattern * 800 + np.random.normal(0, 50, hours)
    ghi = np.maximum(0, ghi)
    
    # Temperature with seasonal variation
    temp_seasonal = 20 + 15 * np.cos(2 * np.pi * (day_of_year - 172) / 365)
    temp_daily = 5 * np.cos(2 * np.pi * (hour_of_day - 15) / 24)
    temperature = temp_seasonal + temp_daily + np.random.normal(0, 2, hours)
    
    # Wind speed
    wind_speed = 3 + 2 * np.random.exponential(1, hours)
    wind_speed = np.minimum(wind_speed, 20)  # Cap at 20 m/s
    
    return {
        'GHI': ghi,
        'Temperature': temperature,
        'WindSpeed': wind_speed
    }

# Generate synthetic data
weather_data = create_synthetic_weather_data()

print("\nWeather Data Summary:")
print(f"Annual GHI: {np.sum(weather_data['GHI']):.0f} Wh/m²")
print(f"Average Temperature: {np.mean(weather_data['Temperature']):.1f}°C")
print(f"Average Wind Speed: {np.mean(weather_data['WindSpeed']):.1f} m/s")

### Running the PV Simulation

Now let's execute the solar PV simulation:

In [ ]:
# Execute the PV simulation
try:
    pv.execute()
    
    # Extract key results
    annual_energy = pv.Outputs.annual_energy  # kWh
    capacity_factor = pv.Outputs.capacity_factor  # %
    monthly_energy = pv.Outputs.monthly_energy  # kWh/month
    hourly_gen = pv.Outputs.gen  # hourly generation (kW)
    
    print("\nPV System Simulation Results:")
    print(f"Annual Energy Production: {annual_energy:,.0f} kWh")
    print(f"Capacity Factor: {capacity_factor:.1f}%")
    print(f"Specific Yield: {annual_energy/pv.SystemDesign.system_capacity:.0f} kWh/kW")
    
    # Monthly breakdown
    months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
              'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    
    print("\nMonthly Energy Production:")
    for i, month in enumerate(months):
        print(f"{month}: {monthly_energy[i]:,.0f} kWh")
    
    # Plot monthly generation
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    plt.bar(months, monthly_energy, color='orange', alpha=0.7)
    plt.title('Monthly PV Generation')
    plt.ylabel('Energy (kWh)')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    
    # Plot first week of hourly generation
    plt.subplot(1, 2, 2)
    week_hours = range(24 * 7)  # First week
    plt.plot(week_hours, hourly_gen[:24*7], 'b-', linewidth=1.5)
    plt.title('First Week Hourly Generation')
    plt.xlabel('Hour')
    plt.ylabel('Power (kW)')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Simulation failed: {e}")
    print("This might be due to missing weather data or incorrect model configuration.")

## 5. Battery Storage Integration {#battery-storage}

Let's add battery storage to our PV system:

In [ ]:
# Create a PV+Battery system
pv_batt = PV.default("FlatPlatePVSingleOwner")
battery = Battery.from_existing(pv_batt)

# PV system configuration (same as before)
pv_batt.SystemDesign.system_capacity = 10.0
pv_batt.SystemDesign.tilt = 30.0
pv_batt.SystemDesign.azimuth = 180.0

# Battery system configuration
battery.BatterySystem.batt_simple_enable = 1  # Enable simple battery model
battery.BatterySystem.batt_simple_kwh = 13.5   # Battery capacity (kWh) - Tesla Powerwall size
battery.BatterySystem.batt_simple_kw = 5.0     # Battery power rating (kW)
battery.BatterySystem.batt_simple_chemistry = 1  # Lithium-ion

# Battery dispatch strategy
battery.BatteryDispatch.batt_dispatch_choice = 3  # Automated economic dispatch
battery.BatteryDispatch.batt_dispatch_auto_can_charge = 1  # Can charge from grid
battery.BatteryDispatch.batt_dispatch_auto_can_clipcharge = 1  # Can charge from clipped PV

# Economic dispatch parameters
battery.ElectricityRates.ur_enable_net_metering = 1  # Enable net metering
battery.ElectricityRates.ur_nm_yearend_sell_rate = 0.03  # Net metering sell rate ($/kWh)

# Time-of-use rates to encourage battery dispatch
# Simplified TOU structure: higher rates in evening peak (5-9 PM)
tou_periods = [1] * 8760  # Start with all off-peak
for hour in range(8760):
    hour_of_day = hour % 24
    if 17 <= hour_of_day < 21:  # 5 PM to 9 PM
        tou_periods[hour] = 2  # Peak period

battery.ElectricityRates.ur_tou_sched_weekend = tou_periods
battery.ElectricityRates.ur_tou_sched_weekday = tou_periods

# Rate structure
battery.ElectricityRates.ur_tou_mat = [
    [1, 1, 0.12, 0.02],  # Period 1: Off-peak rate
    [1, 2, 0.28, 0.02],  # Period 2: Peak rate
]

print("Battery Storage Configuration:")
print(f"Battery Capacity: {battery.BatterySystem.batt_simple_kwh} kWh")
print(f"Battery Power: {battery.BatterySystem.batt_simple_kw} kW")
print(f"Chemistry: {'Lithium-ion' if battery.BatterySystem.batt_simple_chemistry == 1 else 'Lead-acid'}")
print(f"Dispatch Strategy: {'Economic' if battery.BatteryDispatch.batt_dispatch_choice == 3 else 'Manual'}")

In [ ]:
# Run PV+Battery simulation
try:
    # Execute both models
    pv_batt.execute()
    battery.execute()
    
    # Extract battery results
    batt_annual_discharge = battery.Outputs.batt_annual_discharge_energy  # kWh
    batt_annual_charge = battery.Outputs.batt_annual_charge_energy  # kWh
    batt_cycles = battery.Outputs.batt_cycles  # Annual equivalent cycles
    batt_power = battery.Outputs.batt_power  # Hourly battery power (kW)
    batt_soc = battery.Outputs.batt_SOC  # Hourly state of charge (%)
    
    print("\nBattery Performance Results:")
    print(f"Annual Discharge Energy: {batt_annual_discharge:,.0f} kWh")
    print(f"Annual Charge Energy: {batt_annual_charge:,.0f} kWh")
    print(f"Round-trip Efficiency: {(batt_annual_discharge/batt_annual_charge)*100:.1f}%")
    print(f"Annual Equivalent Cycles: {batt_cycles:.1f}")
    
    # System-level metrics
    pv_generation = pv_batt.Outputs.annual_energy
    grid_import = np.sum(np.maximum(0, -np.array(batt_power)))  # Approximate
    
    print(f"\nSystem Integration:")
    print(f"PV Generation: {pv_generation:,.0f} kWh")
    print(f"Battery Utilization: {(batt_annual_discharge/(battery.BatterySystem.batt_simple_kwh*365))*100:.1f}% of daily capacity")
    
    # Visualize battery operation
    fig, axes = plt.subplots(3, 1, figsize=(12, 10))
    
    # First week analysis
    week_hours = range(24 * 7)
    
    # PV generation and battery power
    axes[0].plot(week_hours, pv_batt.Outputs.gen[:24*7], 'orange', label='PV Generation', linewidth=2)
    axes[0].bar(week_hours, batt_power[:24*7], alpha=0.6, 
                color=['red' if x < 0 else 'green' for x in batt_power[:24*7]], 
                label='Battery Power (+ discharge, - charge)')
    axes[0].set_ylabel('Power (kW)')
    axes[0].set_title('First Week: PV Generation and Battery Operation')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Battery state of charge
    axes[1].plot(week_hours, batt_soc[:24*7], 'blue', linewidth=2)
    axes[1].set_ylabel('SOC (%)')
    axes[1].set_title('Battery State of Charge')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0, 100])
    
    # Daily cycle analysis
    daily_discharge = []
    daily_charge = []
    for day in range(30):  # First 30 days
        day_start = day * 24
        day_end = (day + 1) * 24
        day_power = batt_power[day_start:day_end]
        daily_discharge.append(np.sum(np.maximum(0, day_power)))
        daily_charge.append(-np.sum(np.minimum(0, day_power)))
    
    axes[2].bar(range(30), daily_discharge, alpha=0.7, color='green', label='Daily Discharge')
    axes[2].bar(range(30), [-x for x in daily_charge], alpha=0.7, color='red', label='Daily Charge')
    axes[2].set_xlabel('Day')
    axes[2].set_ylabel('Energy (kWh)')
    axes[2].set_title('First 30 Days: Daily Battery Energy Flows')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"PV+Battery simulation failed: {e}")
    print("Check model configuration and data compatibility.")

## 6. Financial Analysis {#financial}

PySAM provides comprehensive financial modeling capabilities:

In [ ]:
# Create financial models for PV and PV+Battery systems
from PySAM import Singleowner

# Financial model for PV-only system
financial_pv = Singleowner.from_existing(pv_batt)

# System costs
system_size_kw = pv_batt.SystemDesign.system_capacity
pv_cost_per_kw = 2500  # $2.50/W installed cost (2024 residential average)
battery_cost_per_kwh = 1000  # $1000/kWh for battery

# PV system costs
total_pv_cost = system_size_kw * 1000 * pv_cost_per_kw / 1000  # Convert to total cost
battery_cost = battery.BatterySystem.batt_simple_kwh * battery_cost_per_kwh

# Set financial parameters
financial_pv.Revenue.ppa_price_input = [0.12] * 25  # Electricity price escalation ($/kWh)
financial_pv.FinancialParameters.analysis_period = 25  # Years
financial_pv.FinancialParameters.debt_percent = 0  # All equity financing
financial_pv.FinancialParameters.discount_rate = 6.0  # Discount rate (%)
financial_pv.FinancialParameters.federal_tax_rate = 22  # Federal tax rate (%)
financial_pv.FinancialParameters.state_tax_rate = 7   # State tax rate (%)
financial_pv.FinancialParameters.property_tax_rate = 0  # Property tax rate (%)
financial_pv.FinancialParameters.insurance_rate = 0.5  # Insurance rate (%)
financial_pv.FinancialParameters.inflation_rate = 2.5  # Inflation rate (%)

# System costs (PV only)
financial_pv.SystemCosts.total_installed_cost = total_pv_cost

# O&M costs
financial_pv.SystemCosts.om_fixed = [25] * 25  # Fixed O&M ($/kW/year)
financial_pv.SystemCosts.om_production = [0] * 25  # Variable O&M ($/MWh)

# Incentives
financial_pv.TaxCreditIncentives.itc_fed_amount = [30] * 25  # Federal ITC (%)
financial_pv.TaxCreditIncentives.itc_fed_amount_deprbas_fed = 1
financial_pv.TaxCreditIncentives.itc_fed_amount_deprbas_sta = 1

print("Financial Model Configuration:")
print(f"PV System Cost: ${total_pv_cost:,.0f} (${pv_cost_per_kw:.2f}/W)")
print(f"Battery Cost: ${battery_cost:,.0f}")
print(f"Analysis Period: {financial_pv.FinancialParameters.analysis_period} years")
print(f"Discount Rate: {financial_pv.FinancialParameters.discount_rate}%")
print(f"Federal ITC: {financial_pv.TaxCreditIncentives.itc_fed_amount[0]}%")


In [ ]:
# Run financial analysis
try:
    financial_pv.execute()
    
    # Extract financial results
    npv = financial_pv.Outputs.npv  # Net present value
    lcoe_nom = financial_pv.Outputs.lcoe_nom  # Nominal LCOE ($/kWh)
    lcoe_real = financial_pv.Outputs.lcoe_real  # Real LCOE ($/kWh)
    payback = financial_pv.Outputs.payback  # Simple payback period (years)
    irr = financial_pv.Outputs.project_return_aftertax_irr  # Internal rate of return (%)
    
    # Cash flows
    cf_after_tax = financial_pv.Outputs.cf_after_tax_cash_flow
    cf_cumulative = np.cumsum(cf_after_tax)
    
    print("\nFinancial Analysis Results:")
    print(f"Net Present Value: ${npv:,.0f}")
    print(f"Levelized Cost of Energy (Real): ${lcoe_real:.3f}/kWh")
    print(f"Levelized Cost of Energy (Nominal): ${lcoe_nom:.3f}/kWh")
    print(f"Simple Payback Period: {payback:.1f} years")
    print(f"Internal Rate of Return: {irr:.1f}%")
    
    # Compare to utility rates
    avg_utility_rate = 0.16  # Average residential rate ($/kWh)
    savings_per_kwh = avg_utility_rate - lcoe_real
    annual_savings = annual_energy * savings_per_kwh
    
    print(f"\nEconomic Comparison:")
    print(f"Average Utility Rate: ${avg_utility_rate:.3f}/kWh")
    print(f"Solar LCOE Savings: ${savings_per_kwh:.3f}/kWh ({savings_per_kwh/avg_utility_rate*100:.1f}% reduction)")
    print(f"Annual Dollar Savings: ${annual_savings:,.0f}")
    print(f"25-Year Cumulative Savings: ${annual_savings*25:,.0f}")
    
    # Visualize financial results
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Cash flow over time
    years = range(len(cf_after_tax))
    axes[0,0].bar(years, cf_after_tax, alpha=0.7, color='blue')
    axes[0,0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    axes[0,0].set_title('Annual After-Tax Cash Flows')
    axes[0,0].set_xlabel('Year')
    axes[0,0].set_ylabel('Cash Flow ($)')
    axes[0,0].grid(True, alpha=0.3)
    
    # Cumulative cash flow
    axes[0,1].plot(years, cf_cumulative, 'g-', linewidth=3)
    axes[0,1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    axes[0,1].set_title('Cumulative Cash Flow')
    axes[0,1].set_xlabel('Year')
    axes[0,1].set_ylabel('Cumulative Cash Flow ($)')
    axes[0,1].grid(True, alpha=0.3)
    
    # Cost breakdown
    cost_categories = ['PV System', 'O&M (25 yr)', 'Insurance (25 yr)']
    total_om = np.sum(financial_pv.SystemCosts.om_fixed) * system_size_kw
    total_insurance = total_pv_cost * (financial_pv.FinancialParameters.insurance_rate/100) * 25
    costs = [total_pv_cost, total_om, total_insurance]
    
    axes[1,0].pie(costs, labels=cost_categories, autopct='%1.1f%%', startangle=90)
    axes[1,0].set_title('Total Cost Breakdown')
    
    # LCOE sensitivity analysis
    discount_rates = np.linspace(3, 10, 8)
    lcoe_sensitivity = []
    
    for rate in discount_rates:
        temp_financial = Singleowner.from_existing(financial_pv)
        temp_financial.FinancialParameters.discount_rate = rate
        try:
            temp_financial.execute()
            lcoe_sensitivity.append(temp_financial.Outputs.lcoe_real)
        except:
            lcoe_sensitivity.append(np.nan)
    
    axes[1,1].plot(discount_rates, lcoe_sensitivity, 'ro-', linewidth=2, markersize=6)
    axes[1,1].axhline(y=avg_utility_rate, color='orange', linestyle='--', 
                      label=f'Utility Rate (${avg_utility_rate:.3f})', linewidth=2)
    axes[1,1].set_title('LCOE vs Discount Rate Sensitivity')
    axes[1,1].set_xlabel('Discount Rate (%)')
    axes[1,1].set_ylabel('LCOE ($/kWh)')
    axes[1,1].grid(True, alpha=0.3)
    axes[1,1].legend()
    
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Financial analysis failed: {e}")

## 7. Wind Power Systems {#wind}

PySAM also supports wind power modeling:

In [ ]:
# Create a wind power system model
wind = Wind.default("WindPowerSingleOwner")

# Wind turbine specifications (example: 2.5 MW turbine)
wind.Turbine.wind_turbine_rotor_diameter = 100  # meters
wind.Turbine.wind_turbine_hub_ht = 80  # meters
wind.Turbine.wind_turbine_max_cp = 0.45  # Maximum power coefficient

# Wind farm configuration
wind.Farm.system_capacity = 50000  # 50 MW wind farm (20 x 2.5 MW turbines)
wind.Farm.wind_farm_xCoordinates = [i * 400 for i in range(20)]  # Turbine x-coordinates (m)
wind.Farm.wind_farm_yCoordinates = [0] * 20  # Turbine y-coordinates (m)

# Losses and availability
wind.Losses.avail_bop_loss = 3.0  # Balance of plant losses (%)
wind.Losses.avail_grid_loss = 1.0  # Grid losses (%)
wind.Losses.avail_turb_loss = 6.0  # Turbine losses (%)
wind.Losses.ops_env_loss = 0.0  # Environmental losses (%)
wind.Losses.ops_grid_loss = 0.0  # Grid operational losses (%)
wind.Losses.ops_load_loss = 0.0  # Load losses (%)
wind.Losses.ops_strategies_loss = 0.0  # Operational strategies losses (%)
wind.Losses.turb_generic_loss = 16.7  # Generic turbine losses (%)

# Create synthetic wind resource data
def create_wind_resource():
    """Create synthetic wind resource data"""
    hours = 8760
    
    # Seasonal wind pattern (higher in winter)
    time = np.arange(hours)
    day_of_year = (time // 24) % 365
    hour_of_day = time % 24
    
    # Base wind speed with seasonal variation
    seasonal_factor = 1 + 0.4 * np.cos(2 * np.pi * (day_of_year - 1) / 365)  # Higher in winter
    daily_variation = 0.2 * np.sin(2 * np.pi * hour_of_day / 24)  # Slight daily pattern
    
    # Weibull distribution parameters
    mean_wind = 7.5 + 2.0 * seasonal_factor + daily_variation
    wind_speeds = np.random.weibull(2.0, hours) * mean_wind
    wind_speeds = np.maximum(0, wind_speeds)  # No negative winds
    wind_speeds = np.minimum(wind_speeds, 25)  # Cap at 25 m/s
    
    # Wind direction (predominant westerly)
    wind_direction = np.random.normal(270, 45, hours) % 360
    
    return wind_speeds, wind_direction

wind_speeds, wind_directions = create_wind_resource()

# Set wind resource (simplified - normally would use weather file)
# wind.Resource.wind_resource_data would be set from file
# For this demo, we'll use the default resource

print("Wind Farm Configuration:")
print(f"Total Capacity: {wind.Farm.system_capacity/1000:.1f} MW")
print(f"Number of Turbines: {len(wind.Farm.wind_farm_xCoordinates)}")
print(f"Turbine Capacity: {wind.Farm.system_capacity/len(wind.Farm.wind_farm_xCoordinates)/1000:.1f} MW each")
print(f"Rotor Diameter: {wind.Turbine.wind_turbine_rotor_diameter} m")
print(f"Hub Height: {wind.Turbine.wind_turbine_hub_ht} m")
print(f"\nSynthetic Wind Resource:")
print(f"Average Wind Speed: {np.mean(wind_speeds):.1f} m/s")
print(f"Wind Speed Std Dev: {np.std(wind_speeds):.1f} m/s")

In [ ]:
# Run wind simulation (if weather data is available)
try:
    wind.execute()
    
    # Extract wind results
    wind_annual_energy = wind.Outputs.annual_energy
    wind_capacity_factor = wind.Outputs.capacity_factor
    wind_monthly_energy = wind.Outputs.monthly_energy
    
    print("\nWind Farm Results:")
    print(f"Annual Energy: {wind_annual_energy:,.0f} MWh")
    print(f"Capacity Factor: {wind_capacity_factor:.1f}%")
    print(f"Specific Yield: {wind_annual_energy*1000/wind.Farm.system_capacity:.0f} kWh/kW")
    
    # Compare wind and solar capacity factors
    technologies = ['Solar PV (10 kW)', f'Wind Farm ({wind.Farm.system_capacity/1000:.0f} MW)']
    capacity_factors = [capacity_factor, wind_capacity_factor]
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    colors = ['orange', 'skyblue']
    bars = plt.bar(technologies, capacity_factors, color=colors, alpha=0.7)
    plt.title('Capacity Factor Comparison')
    plt.ylabel('Capacity Factor (%)')
    plt.ylim([0, max(capacity_factors) * 1.2])
    
    # Add value labels on bars
    for bar, cf in zip(bars, capacity_factors):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{cf:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    
    # Wind speed distribution
    plt.subplot(1, 2, 2)
    plt.hist(wind_speeds, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    plt.title('Wind Speed Distribution')
    plt.xlabel('Wind Speed (m/s)')
    plt.ylabel('Frequency')
    plt.axvline(np.mean(wind_speeds), color='red', linestyle='--', 
                label=f'Mean: {np.mean(wind_speeds):.1f} m/s')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Wind simulation failed: {e}")
    print("This is likely due to missing or incompatible weather data.")
    
    # Show wind resource analysis instead
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.hist(wind_speeds, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    plt.title('Synthetic Wind Speed Distribution')
    plt.xlabel('Wind Speed (m/s)')
    plt.ylabel('Frequency')
    plt.axvline(np.mean(wind_speeds), color='red', linestyle='--', 
                label=f'Mean: {np.mean(wind_speeds):.1f} m/s')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    # Wind power curve estimation
    cut_in = 3
    rated_speed = 12
    cut_out = 25
    
    wind_speeds_curve = np.linspace(0, 30, 100)
    power_curve = np.zeros_like(wind_speeds_curve)
    
    for i, ws in enumerate(wind_speeds_curve):
        if ws < cut_in or ws > cut_out:
            power_curve[i] = 0
        elif ws <= rated_speed:
            power_curve[i] = (ws - cut_in)**3 / (rated_speed - cut_in)**3
        else:
            power_curve[i] = 1.0
    
    plt.plot(wind_speeds_curve, power_curve, 'b-', linewidth=2, label='Typical Power Curve')
    plt.axvline(np.mean(wind_speeds), color='red', linestyle='--', 
                label=f'Site Mean Wind Speed')
    plt.title('Typical Wind Turbine Power Curve')
    plt.xlabel('Wind Speed (m/s)')
    plt.ylabel('Normalized Power Output')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 8. Weather Data Integration {#weather}

PySAM provides tools for working with weather data:

In [ ]:
# Weather data analysis and visualization
import PySAM.ResourceTools as ResourceTools

# Example of weather data analysis (using synthetic data for demo)
def analyze_solar_resource():
    """Analyze solar resource characteristics"""
    
    # Generate synthetic TMY-like data
    hours = 8760
    time = np.arange(hours)
    day_of_year = (time // 24) % 365
    hour_of_day = time % 24
    
    # Solar irradiance components
    solar_elevation = np.maximum(0, 90 * np.cos(2 * np.pi * (day_of_year - 172) / 365) * 
                                 np.cos(2 * np.pi * (hour_of_day - 12) / 24))
    
    # GHI with weather variability
    clear_sky_ghi = 1000 * np.sin(np.radians(solar_elevation))
    cloud_factor = np.random.beta(2, 2, hours)  # Random cloud cover
    ghi = clear_sky_ghi * cloud_factor
    ghi = np.maximum(0, ghi)
    
    # DNI and DHI estimation
    clearness_index = ghi / np.maximum(1, clear_sky_ghi)
    dni = ghi * (clearness_index > 0.6) * clearness_index  # Simplified DNI
    dhi = ghi - dni * np.cos(np.radians(90 - solar_elevation))  # Simplified DHI
    dhi = np.maximum(0, dhi)
    
    # Temperature
    temp_seasonal = 20 + 15 * np.cos(2 * np.pi * (day_of_year - 172) / 365)
    temp_daily = 8 * np.cos(2 * np.pi * (hour_of_day - 15) / 24)
    temperature = temp_seasonal + temp_daily + np.random.normal(0, 3, hours)
    
    # Wind speed
    wind_speed = 2 + 4 * np.random.exponential(1, hours)
    wind_speed = np.minimum(wind_speed, 15)
    
    return {
        'GHI': ghi,
        'DNI': dni,
        'DHI': dhi,
        'Temperature': temperature,
        'WindSpeed': wind_speed,
        'SolarElevation': solar_elevation
    }

# Generate and analyze weather data
weather = analyze_solar_resource()

# Calculate key solar resource metrics
annual_ghi = np.sum(weather['GHI']) / 1000  # kWh/m²/year
annual_dni = np.sum(weather['DNI']) / 1000  # kWh/m²/year
avg_temp = np.mean(weather['Temperature'])
avg_wind = np.mean(weather['WindSpeed'])

print("Solar Resource Analysis:")
print(f"Annual GHI: {annual_ghi:.0f} kWh/m²/year")
print(f"Annual DNI: {annual_dni:.0f} kWh/m²/year")
print(f"Average Temperature: {avg_temp:.1f}°C")
print(f"Average Wind Speed: {avg_wind:.1f} m/s")

# Monthly resource analysis
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

monthly_ghi = []
monthly_temp = []
for month in range(12):
    if month < 11:
        days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    else:
        days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    
    # Approximate monthly data
    start_day = sum(days_in_month[:month])
    end_day = start_day + days_in_month[month]
    start_hour = start_day * 24
    end_hour = min(end_day * 24, 8760)
    
    monthly_ghi.append(np.sum(weather['GHI'][start_hour:end_hour]) / 1000)
    monthly_temp.append(np.mean(weather['Temperature'][start_hour:end_hour]))

# Visualize weather data
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Monthly GHI
axes[0,0].bar(months, monthly_ghi, color='orange', alpha=0.7)
axes[0,0].set_title('Monthly Global Horizontal Irradiance')
axes[0,0].set_ylabel('GHI (kWh/m²/month)')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].grid(True, alpha=0.3)

# Monthly temperature
axes[0,1].plot(months, monthly_temp, 'ro-', linewidth=2, markersize=6)
axes[0,1].set_title('Monthly Average Temperature')
axes[0,1].set_ylabel('Temperature (°C)')
axes[0,1].tick_params(axis='x', rotation=45)
axes[0,1].grid(True, alpha=0.3)

# Daily irradiance pattern (summer day)
summer_day = weather['GHI'][4000:4024]  # Approximate summer day
axes[1,0].plot(range(24), summer_day, 'b-', linewidth=2, marker='o')
axes[1,0].set_title('Typical Summer Day GHI Profile')
axes[1,0].set_xlabel('Hour of Day')
axes[1,0].set_ylabel('GHI (W/m²)')
axes[1,0].grid(True, alpha=0.3)
axes[1,0].set_xticks(range(0, 25, 4))

# GHI vs Temperature scatter plot
sample_indices = np.random.choice(8760, 1000, replace=False)  # Sample for performance
scatter = axes[1,1].scatter(weather['Temperature'][sample_indices], 
                           weather['GHI'][sample_indices], 
                           c=weather['WindSpeed'][sample_indices], 
                           alpha=0.6, cmap='coolwarm')
axes[1,1].set_title('GHI vs Temperature (colored by wind speed)')
axes[1,1].set_xlabel('Temperature (°C)')
axes[1,1].set_ylabel('GHI (W/m²)')
axes[1,1].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[1,1], label='Wind Speed (m/s)')

plt.tight_layout()
plt.show()

print(f"\nResource Quality Assessment:")
ghi_quality = "Excellent" if annual_ghi > 2000 else "Good" if annual_ghi > 1500 else "Moderate"
print(f"GHI Resource Quality: {ghi_quality}")
print(f"Peak Month: {months[np.argmax(monthly_ghi)]} ({max(monthly_ghi):.0f} kWh/m²)")
print(f"Low Month: {months[np.argmin(monthly_ghi)]} ({min(monthly_ghi):.0f} kWh/m²)")
print(f"Seasonal Variability: {(max(monthly_ghi)/min(monthly_ghi)):.1f}x")

## 9. Sensitivity Analysis {#sensitivity}

PySAM enables comprehensive sensitivity and parametric studies:

In [ ]:
# Sensitivity analysis for key parameters
def run_sensitivity_analysis():
    """Run sensitivity analysis on key PV system parameters"""
    
    # Base case model
    base_pv = PV.default("FlatPlatePVResidential")
    base_financial = Singleowner.from_existing(base_pv)
    
    # Base configuration
    base_pv.SystemDesign.system_capacity = 8.0
    base_pv.SystemDesign.tilt = 30
    base_pv.SystemDesign.azimuth = 180
    
    base_financial.SystemCosts.total_installed_cost = 20000
    base_financial.FinancialParameters.discount_rate = 6.0
    base_financial.TaxCreditIncentives.itc_fed_amount = [30] * 25
    
    # Parameters to analyze
    sensitivity_params = {
        'System Size (kW)': {
            'values': np.linspace(4, 12, 9),
            'param_path': ['SystemDesign', 'system_capacity'],
            'cost_scaling': True  # Scale cost with size
        },
        'Tilt Angle (°)': {
            'values': np.linspace(0, 50, 11),
            'param_path': ['SystemDesign', 'tilt'],
            'cost_scaling': False
        },
        'System Cost ($/kW)': {
            'values': np.linspace(2000, 4000, 11),
            'param_path': ['SystemCosts', 'total_installed_cost'],
            'cost_scaling': 'direct'  # Direct cost parameter
        },
        'Discount Rate (%)': {
            'values': np.linspace(3, 10, 8),
            'param_path': ['FinancialParameters', 'discount_rate'],
            'cost_scaling': False
        },
        'Federal ITC (%)': {
            'values': np.linspace(0, 50, 11),
            'param_path': ['TaxCreditIncentives', 'itc_fed_amount'],
            'cost_scaling': False
        }
    }
    
    results = {}
    
    for param_name, config in sensitivity_params.items():
        print(f"Analyzing {param_name}...")
        
        param_results = {
            'values': config['values'],
            'annual_energy': [],
            'lcoe': [],
            'npv': [],
            'payback': []
        }
        
        for value in config['values']:
            try:
                # Create new model instances
                test_pv = PV.from_existing(base_pv)
                test_financial = Singleowner.from_existing(base_financial)
                
                # Set parameter value
                if len(config['param_path']) == 2:
                    group_name, param_attr = config['param_path']
                    
                    if group_name == 'SystemDesign':
                        setattr(getattr(test_pv, group_name), param_attr, value)
                    else:
                        setattr(getattr(test_financial, group_name), param_attr, 
                               [value] * 25 if param_attr == 'itc_fed_amount' else value)
                
                # Handle cost scaling
                if config['cost_scaling'] == True:  # Scale with system size
                    cost_per_kw = 20000 / 8.0  # Base cost per kW
                    test_financial.SystemCosts.total_installed_cost = value * cost_per_kw
                elif config['cost_scaling'] == 'direct':  # Direct cost parameter
                    test_financial.SystemCosts.total_installed_cost = value * test_pv.SystemDesign.system_capacity
                
                # Run simulations
                test_pv.execute()
                test_financial.execute()
                
                # Collect results
                param_results['annual_energy'].append(test_pv.Outputs.annual_energy)
                param_results['lcoe'].append(test_financial.Outputs.lcoe_real)
                param_results['npv'].append(test_financial.Outputs.npv)
                param_results['payback'].append(test_financial.Outputs.payback)
                
            except Exception as e:
                print(f"  Error at {param_name} = {value}: {e}")
                param_results['annual_energy'].append(np.nan)
                param_results['lcoe'].append(np.nan)
                param_results['npv'].append(np.nan)
                param_results['payback'].append(np.nan)
        
        results[param_name] = param_results
    
    return results

# Run the sensitivity analysis
print("Starting comprehensive sensitivity analysis...")
sensitivity_results = run_sensitivity_analysis()
print("Sensitivity analysis complete!")

In [ ]:
# Visualize sensitivity analysis results
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

colors = ['blue', 'green', 'red', 'orange', 'purple']
param_names = list(sensitivity_results.keys())

for i, (param_name, color) in enumerate(zip(param_names, colors)):
    data = sensitivity_results[param_name]
    
    # LCOE sensitivity
    axes[i].plot(data['values'], data['lcoe'], f'{color[0]}o-', linewidth=2, markersize=6)
    axes[i].set_title(f'LCOE vs {param_name}')
    axes[i].set_xlabel(param_name)
    axes[i].set_ylabel('LCOE ($/kWh)')
    axes[i].grid(True, alpha=0.3)
    
    # Add horizontal line for utility rate comparison
    axes[i].axhline(y=0.16, color='red', linestyle='--', alpha=0.7, 
                   label='Utility Rate')
    axes[i].legend()

# Summary plot: tornado chart for LCOE sensitivity
if len(param_names) >= 5:  # Only if we have all parameters
    # Calculate sensitivity ranges
    base_lcoe = 0.08  # Approximate base case
    sensitivity_ranges = []
    param_labels = []
    
    for param_name in param_names:
        data = sensitivity_results[param_name]
        valid_lcoe = [x for x in data['lcoe'] if not np.isnan(x)]
        if valid_lcoe:
            min_lcoe = min(valid_lcoe)
            max_lcoe = max(valid_lcoe)
            sensitivity_ranges.append((min_lcoe - base_lcoe, max_lcoe - base_lcoe))
            param_labels.append(param_name.split(' (')[0])  # Remove units
    
    # Create tornado chart
    axes[5].barh(range(len(param_labels)), [r[1] for r in sensitivity_ranges], 
                 left=[r[0] for r in sensitivity_ranges], color='lightblue', alpha=0.7)
    axes[5].set_yticks(range(len(param_labels)))
    axes[5].set_yticklabels(param_labels)
    axes[5].set_xlabel('LCOE Sensitivity ($/kWh)')
    axes[5].set_title('LCOE Tornado Chart')
    axes[5].axvline(x=0, color='red', linestyle='-', linewidth=2)
    axes[5].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print key sensitivity insights
print("\nSensitivity Analysis Summary:")
for param_name in param_names:
    data = sensitivity_results[param_name]
    valid_lcoe = [x for x in data['lcoe'] if not np.isnan(x)]
    valid_npv = [x for x in data['npv'] if not np.isnan(x)]
    
    if valid_lcoe and valid_npv:
        lcoe_range = max(valid_lcoe) - min(valid_lcoe)
        npv_range = max(valid_npv) - min(valid_npv)
        print(f"{param_name}:")
        print(f"  LCOE Range: ${lcoe_range:.3f}/kWh ({lcoe_range/np.mean(valid_lcoe)*100:.1f}% variation)")
        print(f"  NPV Range: ${npv_range:,.0f} ({npv_range/abs(np.mean(valid_npv))*100:.1f}% variation)")

## 10. Advanced Features {#advanced}

PySAM offers several advanced capabilities for power systems analysis:

In [ ]:
# Advanced feature demonstrations

# 1. Custom dispatch strategies for battery systems
def create_custom_battery_dispatch():
    """Create a custom battery dispatch schedule"""
    
    # Time-of-use based dispatch
    # Charge during low-cost hours (overnight), discharge during high-cost hours (evening)
    dispatch_schedule = []
    
    for hour in range(8760):
        hour_of_day = hour % 24
        day_of_year = hour // 24
        
        # Seasonal adjustment
        is_summer = 150 <= day_of_year <= 270  # Approximate summer months
        
        if is_summer:
            # Summer dispatch (air conditioning load)
            if 1 <= hour_of_day <= 6:  # Overnight charging
                dispatch_schedule.append(-1)  # Charge at full power
            elif 14 <= hour_of_day <= 20:  # Afternoon/evening discharge
                dispatch_schedule.append(1)   # Discharge at full power
            else:
                dispatch_schedule.append(0)   # No dispatch
        else:
            # Winter dispatch (heating load)
            if 2 <= hour_of_day <= 7:   # Overnight charging
                dispatch_schedule.append(-1)
            elif 17 <= hour_of_day <= 21:  # Evening discharge
                dispatch_schedule.append(1)
            else:
                dispatch_schedule.append(0)
    
    return dispatch_schedule

# 2. Multi-technology hybrid system
def create_hybrid_system():
    """Create a PV + Wind + Battery hybrid system"""
    
    # This is a conceptual example - actual implementation would require
    # careful integration of multiple SAM models
    
    system_config = {
        'pv_capacity': 10.0,    # kW
        'wind_capacity': 5.0,   # kW (small wind)
        'battery_energy': 20.0, # kWh
        'battery_power': 8.0,   # kW
        'load_profile': 'residential'  # Load type
    }
    
    return system_config

# 3. Advanced financial structures
def create_ppa_model():
    """Create a Power Purchase Agreement (PPA) financial model"""
    
    ppa_config = {
        'ppa_escalation': 2.5,      # %/year
        'ppa_term': 20,             # years
        'ppa_price_year1': 0.08,    # $/kWh
        'debt_fraction': 70,        # %
        'debt_rate': 4.5,          # %
        'debt_term': 18,           # years
        'equity_return': 12.0,     # % IRR target
    }
    
    return ppa_config

# 4. Monte Carlo uncertainty analysis
def monte_carlo_analysis(n_iterations=100):
    """Perform Monte Carlo analysis on key uncertainties"""
    
    print(f"Running Monte Carlo analysis with {n_iterations} iterations...")
    
    # Define uncertainty distributions
    uncertainties = {
        'solar_resource': {'dist': 'normal', 'mean': 1.0, 'std': 0.1},   # ±10% resource
        'system_cost': {'dist': 'normal', 'mean': 2500, 'std': 250},     # ±$250/kW
        'degradation': {'dist': 'uniform', 'low': 0.4, 'high': 0.8},     # %/year
        'electricity_price': {'dist': 'lognormal', 'mean': 0.15, 'sigma': 0.2},  # $/kWh
    }
    
    # Results storage
    mc_results = {
        'lcoe': [],
        'npv': [],
        'annual_energy': [],
        'payback': []
    }
    
    # Base model
    base_pv = PV.default("FlatPlatePVResidential")
    base_financial = Singleowner.from_existing(base_pv)
    
    base_pv.SystemDesign.system_capacity = 8.0
    base_financial.FinancialParameters.analysis_period = 25
    
    for iteration in range(n_iterations):
        if iteration % 20 == 0:
            print(f"  Iteration {iteration}...")
        
        try:
            # Create model instances
            mc_pv = PV.from_existing(base_pv)
            mc_financial = Singleowner.from_existing(base_financial)
            
            # Sample uncertain parameters
            solar_factor = np.random.normal(1.0, 0.1)
            cost_per_kw = np.random.normal(2500, 250)
            degradation = np.random.uniform(0.4, 0.8)
            elec_price = np.random.lognormal(np.log(0.15), 0.2)
            
            # Apply uncertainties
            # Solar resource scaling would require weather file modification
            # For this demo, we'll adjust system performance
            mc_pv.SystemDesign.losses = 14.08 * (2.0 - solar_factor)  # Inverse relationship
            
            mc_financial.SystemCosts.total_installed_cost = cost_per_kw * mc_pv.SystemDesign.system_capacity
            mc_financial.Degradation.degradation = [degradation] * 25
            mc_financial.Revenue.ppa_price_input = [elec_price] * 25
            
            # Run simulation
            mc_pv.execute()
            mc_financial.execute()
            
            # Store results
            mc_results['lcoe'].append(mc_financial.Outputs.lcoe_real)
            mc_results['npv'].append(mc_financial.Outputs.npv)
            mc_results['annual_energy'].append(mc_pv.Outputs.annual_energy)
            mc_results['payback'].append(mc_financial.Outputs.payback)
            
        except Exception as e:
            # Handle simulation failures
            for key in mc_results.keys():
                mc_results[key].append(np.nan)
    
    print("Monte Carlo analysis complete!")
    return mc_results

# Execute advanced features
print("Advanced Features Demonstration:")
print("\n1. Custom Battery Dispatch Strategy:")
custom_dispatch = create_custom_battery_dispatch()
print(f"   Created custom dispatch schedule with {len(custom_dispatch)} hourly values")
print(f"   Charge hours: {sum(1 for x in custom_dispatch if x < 0)} hours/year")
print(f"   Discharge hours: {sum(1 for x in custom_dispatch if x > 0)} hours/year")

print("\n2. Hybrid System Configuration:")
hybrid_config = create_hybrid_system()
for key, value in hybrid_config.items():
    print(f"   {key}: {value}")

print("\n3. PPA Financial Model:")
ppa_config = create_ppa_model()
for key, value in ppa_config.items():
    print(f"   {key}: {value}")

print("\n4. Monte Carlo Analysis:")
mc_results = monte_carlo_analysis(50)  # Reduced iterations for demo

In [ ]:
# Visualize Monte Carlo results
if mc_results and any(not np.isnan(x) for x in mc_results['lcoe']):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Remove NaN values
    valid_results = {}
    for key in mc_results.keys():
        valid_results[key] = [x for x in mc_results[key] if not np.isnan(x)]
    
    # LCOE distribution
    axes[0,0].hist(valid_results['lcoe'], bins=20, alpha=0.7, color='blue', edgecolor='black')
    axes[0,0].axvline(np.mean(valid_results['lcoe']), color='red', linestyle='--', 
                     label=f'Mean: ${np.mean(valid_results["lcoe"]):.3f}/kWh')
    axes[0,0].set_title('LCOE Distribution')
    axes[0,0].set_xlabel('LCOE ($/kWh)')
    axes[0,0].set_ylabel('Frequency')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # NPV distribution
    axes[0,1].hist(valid_results['npv'], bins=20, alpha=0.7, color='green', edgecolor='black')
    axes[0,1].axvline(np.mean(valid_results['npv']), color='red', linestyle='--', 
                     label=f'Mean: ${np.mean(valid_results["npv"]):,.0f}')
    axes[0,1].axvline(0, color='orange', linestyle=':', label='Break-even')
    axes[0,1].set_title('NPV Distribution')
    axes[0,1].set_xlabel('NPV ($)')
    axes[0,1].set_ylabel('Frequency')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # Payback distribution
    axes[1,0].hist(valid_results['payback'], bins=20, alpha=0.7, color='orange', edgecolor='black')
    axes[1,0].axvline(np.mean(valid_results['payback']), color='red', linestyle='--', 
                     label=f'Mean: {np.mean(valid_results["payback"]):.1f} years')
    axes[1,0].set_title('Payback Period Distribution')
    axes[1,0].set_xlabel('Payback Period (years)')
    axes[1,0].set_ylabel('Frequency')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # Risk assessment
    positive_npv_pct = sum(1 for x in valid_results['npv'] if x > 0) / len(valid_results['npv']) * 100
    payback_under_10_pct = sum(1 for x in valid_results['payback'] if x < 10) / len(valid_results['payback']) * 100
    
    risk_metrics = {
        'Positive NPV': positive_npv_pct,
        'Payback < 10 years': payback_under_10_pct,
        'LCOE < $0.12/kWh': sum(1 for x in valid_results['lcoe'] if x < 0.12) / len(valid_results['lcoe']) * 100
    }
    
    metrics = list(risk_metrics.keys())
    values = list(risk_metrics.values())
    colors = ['green' if v > 70 else 'orange' if v > 50 else 'red' for v in values]
    
    bars = axes[1,1].bar(metrics, values, color=colors, alpha=0.7)
    axes[1,1].set_title('Risk Assessment')
    axes[1,1].set_ylabel('Probability (%)')
    axes[1,1].set_ylim([0, 100])
    axes[1,1].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        axes[1,1].text(bar.get_x() + bar.get_width()/2., height + 1,
                      f'{value:.0f}%', ha='center', va='bottom', fontweight='bold')
    
    axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Statistical summary
    print("\nMonte Carlo Results Summary:")
    print(f"LCOE: ${np.mean(valid_results['lcoe']):.3f} ± ${np.std(valid_results['lcoe']):.3f} /kWh")
    print(f"NPV: ${np.mean(valid_results['npv']):,.0f} ± ${np.std(valid_results['npv']):,.0f}")
    print(f"Payback: {np.mean(valid_results['payback']):.1f} ± {np.std(valid_results['payback']):.1f} years")
    print(f"\nRisk Metrics:")
    for metric, value in risk_metrics.items():
        print(f"{metric}: {value:.0f}% probability")
    
else:
    print("Monte Carlo analysis produced insufficient valid results for visualization.")

## Summary and Next Steps

This notebook has demonstrated the key capabilities of NREL's PySAM (System Advisor Model):

### Key Features Covered
1. **Model Initialization**: Multiple ways to create and configure models
2. **Solar PV Systems**: Comprehensive modeling with performance and financial analysis
3. **Battery Storage**: Integration with PV systems and dispatch optimization
4. **Wind Power**: Wind resource analysis and turbine modeling
5. **Financial Analysis**: LCOE, NPV, payback periods, and various financing structures
6. **Weather Data**: Resource analysis and meteorological data integration
7. **Sensitivity Analysis**: Parametric studies and optimization
8. **Advanced Features**: Custom dispatch, hybrid systems, and Monte Carlo analysis

### Key Benefits of PySAM
- **Validated Models**: NREL's peer-reviewed physics and financial models
- **Comprehensive Coverage**: Wide range of renewable energy technologies
- **Python Integration**: Seamless integration with data science workflows
- **Scalability**: From residential to utility-scale projects
- **Open Source**: Free to use and modify

### Next Steps for Users
1. **Install PySAM**: `pip install NREL-PySAM`
2. **Explore Documentation**: https://nrel-pysam.readthedocs.io/
3. **Try Examples**: https://github.com/NREL/pysam/tree/main/Examples
4. **Join Community**: SAM and PySAM support forums
5. **Integrate with Your Workflow**: Combine with pandas, matplotlib, and other tools

### Additional Resources
- **SAM Desktop Application**: https://sam.nrel.gov/ (GUI version)
- **SAM Documentation**: Comprehensive user guides and tutorials
- **NREL Webinars**: Regular training sessions on SAM and PySAM
- **GitHub Repository**: Source code, issues, and community contributions

PySAM provides a powerful platform for renewable energy analysis, enabling researchers, developers, and analysts to perform sophisticated techno-economic studies with confidence in the underlying models and methodologies.